In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 0-pre — Environment verification
# FIX: inspect.getsource() fails on some PEP 660 editable installs
# (import-hook-based finder doesn't always support linecache lookup).
# Reading the .py files directly via open() avoids this entirely and
# is equally reliable for checking source text.
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, '-c', 'import depthcharge; print(depthcharge.__file__)'],
    capture_output=True, text=True
)
dc_path = result.stdout.strip()
print(f'depthcharge location: {dc_path}')

if 'site-packages' in dc_path:
    print('⚠ Editable install not active — installing now...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-e',
         '/teamspace/studios/this_studio/depthcharge_changes',
         '--break-system-packages'],
        check=True
    )
    print('Done. Please RESTART the kernel and re-run this cell.')
else:
    print('✓ Editable install confirmed')

    dc_root = os.path.dirname(dc_path)  # .../depthcharge_changes/depthcharge

    analytes_path = os.path.join(dc_root, 'transformers', 'analytes.py')
    spectra_path  = os.path.join(dc_root, 'transformers', 'spectra.py')

    with open(analytes_path) as f:
        analytes_src = f.read()
    with open(spectra_path) as f:
        spectra_src = f.read()

    ok1 = 'flash_compatible' in analytes_src
    ok2 = 'new_zeros' in spectra_src

    print(f'  flash_compatible in analytes.py : {"✓" if ok1 else "✗ MISSING"}')
    print(f'  new_zeros in spectra.py         : {"✓" if ok2 else "✗ MISSING"}')

    if not (ok1 and ok2):
        raise RuntimeError(
            f'Branch changes not detected.\n'
            f'  analytes.py path: {analytes_path}\n'
            f'  spectra.py path : {spectra_path}'
        )
    print('\nReady → proceed to Cell 0 (NAR patch)')

depthcharge location: /teamspace/studios/this_studio/depthcharge_changes/depthcharge/__init__.py
✓ Editable install confirmed
  flash_compatible in analytes.py : ✓
  new_zeros in spectra.py         : ✓

Ready → proceed to Cell 0 (NAR patch)


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 0 — NAR Patch (flash_compatible variant)
# FIX: verification block now reads function source via the `dis`/`__code__`
# co_filename + linecache fallback removed — instead we verify behaviorally
# (call the patched function's logic markers via a direct string check on
# its __code__.co_consts / closure, which doesn't depend on inspect at all)
# to avoid the same OSError seen in Cell 0-pre.
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys, warnings
warnings.filterwarnings('ignore')


import torch
from casanovo.denovo.transformers import PeptideDecoder
from casanovo.denovo.model import Spec2Pep
from depthcharge.transformers import AnalyteTransformerDecoder

import depthcharge as _dc
print(f'depthcharge: {_dc.__file__}')
if 'site-packages' in _dc.__file__:
    raise RuntimeError(
        'Editable install not active.\n'
        'Run: pip install -e /teamspace/studios/this_studio/depthcharge_changes --break-system-packages'
    )
print('  ✓ local branch confirmed (not site-packages)\n')

# ── PATCH 1 — PeptideDecoder.embed ────────────────────────────────────
_ar_embed_original = AnalyteTransformerDecoder.embed

def _nar_embed(self, tokens, *args,
               memory,
               memory_key_padding_mask=None,
               memory_mask=None,
               tgt_mask=None,
               flash_compatible=False,
               _orig=_ar_embed_original,
               **kwargs):
    return _orig(
        self, tokens, *args,
        memory=memory,
        memory_key_padding_mask=memory_key_padding_mask,
        memory_mask=memory_mask,
        flash_compatible=True,
        **kwargs,
    )

assert _ar_embed_original is not _nar_embed, (
    'BUG: captured original is already the patch — restart kernel.')
PeptideDecoder.embed = _nar_embed

# ── PATCH 2 — Spec2Pep._forward_step (unchanged) ─────────────────────
def _nar_forward_step(self, batch):
    mzs, ints, precursors, seqs = self._process_batch(batch)
    dev = self.device
    mzs = mzs.to(dev); ints = ints.to(dev); precursors = precursors.to(dev)
    memories, mem_masks = self.encoder(mzs, ints)
    zero_tokens = (torch.zeros_like(seqs.to(dev)) if seqs is not None
                   else torch.zeros((mzs.shape[0], self.max_peptide_len),
                                    dtype=torch.long, device=dev))
    scores = self.decoder(tokens=zero_tokens, memory=memories,
                          memory_key_padding_mask=mem_masks,
                          precursors=precursors)
    return scores, seqs
Spec2Pep._forward_step = _nar_forward_step

# ── PATCH 3 — Spec2Pep.forward (unchanged) ───────────────────────────
def _nar_forward(self, batch): return self._forward_step(batch)
Spec2Pep.forward = _nar_forward

# ── Verification — behavioral, NOT inspect.getsource() ────────────────
# Checks the function's own __defaults__/__code__ structure directly,
# avoiding the linecache/OSError issue entirely.
import inspect as _inspect

_checks = {
    'PeptideDecoder.embed → _nar_embed'           : PeptideDecoder.embed is _nar_embed,
    'Spec2Pep._forward_step → _nar_forward_step'  : Spec2Pep._forward_step is _nar_forward_step,
    'Spec2Pep.forward → _nar_forward'             : Spec2Pep.forward is _nar_forward,
    'flash_compatible named param in _nar_embed'  : 'flash_compatible' in _inspect.signature(_nar_embed).parameters,
    'zero_tokens var in _nar_forward_step code'   : 'zero_tokens' in _nar_forward_step.__code__.co_varnames,
}
print('── NAR Patch Status ──────────────────────────────────────────')
for k, v in _checks.items():
    print(f'  {k:50s}: {"✓" if v else "✗ FAILED"}')
print('──────────────────────────────────────────────────────────────')
if not all(_checks.values()):
    raise RuntimeError('One or more patches failed.')
print('\nNAR patches applied ✓  (flash_compatible variant, duplicate-kwarg bug fixed)')

depthcharge: /teamspace/studios/this_studio/depthcharge_changes/depthcharge/__init__.py
  ✓ local branch confirmed (not site-packages)

── NAR Patch Status ──────────────────────────────────────────
  PeptideDecoder.embed → _nar_embed                 : ✓
  Spec2Pep._forward_step → _nar_forward_step        : ✓
  Spec2Pep.forward → _nar_forward                   : ✓
  flash_compatible named param in _nar_embed        : ✓
  zero_tokens var in _nar_forward_step code         : ✓
──────────────────────────────────────────────────────────────

NAR patches applied ✓  (flash_compatible variant, duplicate-kwarg bug fixed)


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — Setup + Data + Model
# ═══════════════════════════════════════════════════════════════════════
import os, time, threading
import numpy as np, pandas as pd, datetime
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from torch.profiler import profile, ProfilerActivity, schedule, record_function
from torch.nn.attention import SDPBackend, sdpa_kernel
from contextlib import contextmanager
from tqdm import tqdm

WORK_DIR    = '/teamspace/studios/this_studio/nar_profiling'
RESULTS_DIR = '/teamspace/studios/this_studio/profiling_after_depthcharge_changes/results'
os.chdir(WORK_DIR)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Working directory : {os.getcwd()}')
print(f'Results directory : {RESULTS_DIR}')

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME   = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'
TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9 if DEVICE == 'cuda' else 0
BF16_DTYPE     = torch.bfloat16
BF16_SUPPORTED = torch.cuda.is_bf16_supported() if DEVICE == 'cuda' else False

N_SUBSET         = 6000
N_TIMING_SPECTRA = 5000
BATCH_SIZES      = [1, 8, 32, 128, 512]
N_WARMUP_BATCHES = 10
N_PEAKS          = 150
PROF_WARMUP      = 20
PROF_ACTIVE      = 50

def _sync():
    if DEVICE == 'cuda': torch.cuda.synchronize()

@contextmanager
def _bf16_ctx():
    with torch.autocast(device_type='cuda', dtype=BF16_DTYPE):
        yield

def _mark_step():
    if DEVICE == 'cuda':
        torch.compiler.cudagraph_mark_step_begin()

def pad_or_select_to_fixed_peaks(mzs, ints, n=N_PEAKS):
    bs, L = mzs.shape
    if L == n: return mzs, ints, 0
    if L < n:
        return (torch.nn.functional.pad(mzs,  (0, n - L)),
                torch.nn.functional.pad(ints, (0, n - L)), 0)
    idx = ints.topk(n, dim=1).indices
    return mzs.gather(1, idx), ints.gather(1, idx), bs

def _start_gpu_monitor():
    samples, stop = [], threading.Event()
    def _fn():
        import subprocess as sp
        while not stop.is_set():
            r = sp.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used',
                        '--format=csv,noheader,nounits'],
                       capture_output=True, text=True)
            if r.returncode == 0:
                try:
                    u, m = r.stdout.strip().split(', ')
                    samples.append((int(u), float(m)/1024))
                except Exception: pass
            time.sleep(0.5)
    threading.Thread(target=_fn, daemon=True).start()
    return samples, stop

def _detect_attention_kernel(store, label):
    if not (DEVICE == 'cuda' and store.get('avgs')): return f'{label}: no CUDA data'
    flash = next((e for e in store['avgs'] if 'flash_attention' in e.key), None)
    eff   = next((e for e in store['avgs'] if 'efficient_attention' in e.key), None)
    if flash and eff:
        msg = f'{label}: BOTH flash ({flash.count}) + efficient ({eff.count})'
    elif flash:
        msg = f'{label}: FlashAttention ACTIVE ✓  ({flash.key}, {flash.count} calls)'
    elif eff:
        msg = f'{label}: memory-efficient only (NOT Flash)  ({eff.count} calls)'
    else:
        msg = f'{label}: no attention kernel found'
    print(msg); return msg

def _launch_count(store, n_active=PROF_ACTIVE):
    if not store.get('avgs'): return None
    e = next((e for e in store['avgs'] if e.key == 'cudaLaunchKernel'), None)
    return e.count // n_active if e else None

def _compiled_region_count(store):
    if not store.get('avgs'): return None
    return len([e for e in store['avgs'] if 'Torch-Compiled Region' in e.key])

def _save(obj, name):
    path = os.path.join(RESULTS_DIR, name)
    if hasattr(obj, 'savefig'):
        obj.savefig(path, dpi=150, bbox_inches='tight')
    else:
        obj.to_csv(path, index=False)
    print(f'Saved: {path}')
    return path

_tgt = lambda ms: 'MEETS ✓' if ms <= 10 else f'FAILS ({ms:.1f}ms)'

MGF_FILE   = 'multi-enzyme-simple.test.mgf'
SUBSET_MGF = 'subset_profile.mgf'
LANCE_DIR  = '.lance_cache'

if not os.path.exists(MGF_FILE):
    raise FileNotFoundError(f'{MGF_FILE} not found in {WORK_DIR}')
print(f'{MGF_FILE}  ({os.path.getsize(MGF_FILE)/1e6:.1f} MB)')

print('Spectra : 106,933')
print('Charge  : +1 to +8 | +2: 40,614  +3: 39,146')
print('m/z     : 301.2 – 1604.3')
print('Peaks   : 123 avg  (min 6, max 950)')

if not os.path.exists(SUBSET_MGF):
    raise FileNotFoundError(f'{SUBSET_MGF} not found in {WORK_DIR}')
print(f'Reusing: {SUBSET_MGF}  ({os.path.getsize(SUBSET_MGF)/1e6:.1f} MB)')

from casanovo.denovo import ModelRunner
from casanovo.denovo.model import Spec2Pep
from casanovo.denovo.dataloaders import DeNovoDataModule
from casanovo.config import Config
from casanovo.casanovo import _get_model_weights
import appdirs

config = Config(None)
cache_dir = Path(appdirs.user_cache_dir('casanovo', False, opinion=False))
model_path = _get_model_weights(cache_dir)
print(f'Model checkpoint: {model_path}')

runner = ModelRunner(config, model_path)
runner.initialize_tokenizer()
runner.initialize_model(train=False)
model = runner.model.eval().to(DEVICE)
MODEL_MAX_CHARGE = getattr(model, 'max_charge', config.max_charge)

print(f'Model: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params | '
      f'max_peptide_len={model.max_peptide_len} | max_charge={MODEL_MAX_CHARGE}')

import inspect as _inspect
from casanovo.denovo.transformers import PeptideDecoder
assert PeptideDecoder.embed is _nar_embed, 'NAR patch lost — re-run Cell 0!'
ok_fc = 'flash_compatible=True' in _inspect.getsource(_nar_embed)
print(f'NAR patch active ✓ | flash_compatible=True in patch: {"✓" if ok_fc else "✗"}')

_dm = DeNovoDataModule(
    lance_dir=LANCE_DIR,
    test_paths=[SUBSET_MGF],
    eval_batch_size=1,
    tokenizer=runner.tokenizer,
    max_charge=MODEL_MAX_CHARGE,
    n_workers=0,
)
_dm.setup(stage='test', annotated=False)
_b              = next(iter(_dm.predict_dataloader()))
_mz, _it, _pr, _ = model._process_batch(_b)
print(f'First batch mzs={_mz.shape}  precs={_pr.shape} ✓')
print(f'Subset ready: {N_SUBSET} spectra  (timing target: {N_TIMING_SPECTRA})')

print(f'\nDevice : {DEVICE} | {GPU_NAME} | VRAM: {TOTAL_VRAM:.1f} GB')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda} | BF16: {BF16_SUPPORTED}')
print(f'Batch sizes: {BATCH_SIZES}')

Working directory : /teamspace/studios/this_studio/nar_profiling
Results directory : /teamspace/studios/this_studio/profiling_after_depthcharge_changes/results
multi-enzyme-simple.test.mgf  (300.9 MB)
Spectra : 106,933
Charge  : +1 to +8 | +2: 40,614  +3: 39,146
m/z     : 301.2 – 1604.3
Peaks   : 123 avg  (min 6, max 950)
Reusing: subset_profile.mgf  (16.9 MB)


Checkpoint directory not set in ModelRunner, no checkpoint files will be saved.
Configured residue(s) not in model alphabet: M[Oxidation], [Ammonia-loss]-, N[Deamidated], [Carbamyl]-, [Acetyl]-, Q[Deamidated], C[Carbamidomethyl], [+25.980265]-


Model checkpoint: /home/zeus/.cache/casanovo/casanovo_v5_0_0_v5_0_0.ckpt
Model: 47.9M params | max_peptide_len=100 | max_charge=4
NAR patch active ✓ | flash_compatible=True in patch: ✓


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

First batch mzs=torch.Size([1, 42])  precs=torch.Size([1, 3]) ✓
Subset ready: 6000 spectra  (timing target: 5000)

Device : cuda | NVIDIA L4 | VRAM: 23.6 GB
PyTorch: 2.7.1+cu128 | CUDA: 12.8 | BF16: True
Batch sizes: [1, 8, 32, 128, 512]


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — Baseline NAR FP32 Timing (eager, flash_compatible=True via patch)
# FP32 still uses memory-efficient attention (flash requires fp16/bf16),
# but this is now the clean baseline using the new patch mechanism.
# ═══════════════════════════════════════════════════════════════════════
_gpu_s, _gpu_stop = _start_gpu_monitor()
timing_fp32 = {}

for bs in BATCH_SIZES:
    print(f'\n══ FP32 Baseline  batch_size={bs:4d} ══')
    _dm_bs = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                              eval_batch_size=bs, tokenizer=runner.model.tokenizer,
                              max_charge=MODEL_MAX_CHARGE, n_workers=0)
    _dm_bs.setup(stage='test', annotated=False)
    with torch.no_grad():
        for _w, _wb in enumerate(iter(_dm_bs.predict_dataloader())):
            if _w >= N_WARMUP_BATCHES: break
            _wm, _wi, _wp, _ = model._process_batch(_wb)
            _wm=_wm.to(DEVICE); _wi=_wi.to(DEVICE); _wp=_wp.to(DEVICE)
            _wme, _wmk = model.encoder(_wm, _wi)
            model.decoder(tokens=torch.zeros((_wm.shape[0], model.max_peptide_len),
                          dtype=torch.long, device=DEVICE),
                          memory=_wme, memory_key_padding_mask=_wmk, precursors=_wp)
    _sync()

    _t = {k: [] for k in ['fetch','h2d','enc','nar','write','total','tp']}
    _it = iter(_dm_bs.predict_dataloader()); n_spec = 0
    pbar = tqdm(total=N_TIMING_SPECTRA, desc=f'  bs={bs}', unit='spec')
    while n_spec < N_TIMING_SPECTRA:
        _sync(); t0 = time.perf_counter()
        try: batch = next(_it)
        except StopIteration: _it = iter(_dm_bs.predict_dataloader()); batch = next(_it)
        t_fetch = (time.perf_counter()-t0)*1000

        _sync(); t0 = time.perf_counter()
        mzs,ints,precs,_ = model._process_batch(batch)
        mzs=mzs.to(DEVICE); ints=ints.to(DEVICE); precs=precs.to(DEVICE)
        _sync(); t_h2d=(time.perf_counter()-t0)*1000; ab=mzs.shape[0]

        with torch.no_grad():
            _sync(); t0=time.perf_counter()
            mem,mmk = model.encoder(mzs,ints)
            _sync(); t_enc=(time.perf_counter()-t0)*1000
            zt = torch.zeros((ab,model.max_peptide_len),dtype=torch.long,device=DEVICE)
            _sync(); t0=time.perf_counter()
            scores = model.decoder(tokens=zt,memory=mem,
                                   memory_key_padding_mask=mmk,precursors=precs)
            _sync(); t_nar=(time.perf_counter()-t0)*1000

        t0=time.perf_counter()
        pred=scores.argmax(dim=-1).cpu(); _=[{'tokens':t.tolist()} for t in pred]
        t_write=(time.perf_counter()-t0)*1000
        tt=t_fetch+t_h2d+t_enc+t_nar+t_write
        for k,v in zip(['fetch','h2d','enc','nar','write','total','tp'],
                       [t_fetch/ab,t_h2d/ab,t_enc/ab,t_nar/ab,t_write/ab,tt/ab,ab/(tt/1000)]):
            _t[k].append(v)
        n_spec+=ab; pbar.update(ab)
        if n_spec>=N_TIMING_SPECTRA: break
    pbar.close()

    p = lambda a,q: float(np.percentile(a,q))
    timing_fp32[bs] = {'n_spec':n_spec,'fetch':np.mean(_t['fetch']),'h2d':np.mean(_t['h2d']),
        'enc':np.mean(_t['enc']),'nar':np.mean(_t['nar']),'write':np.mean(_t['write']),
        'total':np.mean(_t['total']),'p50':p(_t['total'],50),'p95':p(_t['total'],95),
        'tp':np.mean(_t['tp']),'raw':_t}
    s=timing_fp32[bs]
    print(f'  total={s["total"]:.2f}ms  enc={s["enc"]:.2f}ms  nar={s["nar"]:.2f}ms  tp={s["tp"]:.1f}spec/s')
    if DEVICE=='cuda': torch.cuda.empty_cache()

_gpu_stop.set(); time.sleep(1.0)
gpu_util_fp32=np.mean([s[0] for s in _gpu_s]) if _gpu_s else 0
gpu_vram_fp32=np.max([s[1] for s in _gpu_s]) if _gpu_s else 0

b1=timing_fp32[1]; rb=b1['raw']
df_stage_fp32 = pd.DataFrame([
    {'Stage':'DataLoader fetch','mean_ms':b1['fetch'],'p50_ms':np.percentile(rb['fetch'],50),'p95_ms':np.percentile(rb['fetch'],95)},
    {'Stage':'H2D transfer',    'mean_ms':b1['h2d'],  'p50_ms':np.percentile(rb['h2d'],50),  'p95_ms':np.percentile(rb['h2d'],95)},
    {'Stage':'SpectrumEncoder', 'mean_ms':b1['enc'],  'p50_ms':np.percentile(rb['enc'],50),  'p95_ms':np.percentile(rb['enc'],95)},
    {'Stage':'NAR Decoder',     'mean_ms':b1['nar'],  'p50_ms':np.percentile(rb['nar'],50),  'p95_ms':np.percentile(rb['nar'],95)},
    {'Stage':'Output write',    'mean_ms':b1['write'],'p50_ms':np.percentile(rb['write'],50),'p95_ms':np.percentile(rb['write'],95)},
    {'Stage':'TOTAL',           'mean_ms':b1['total'],'p50_ms':b1['p50'],                   'p95_ms':b1['p95']},
]).round(3)
df_tp_fp32 = pd.DataFrame([{'batch_size':bs,'total_ms':timing_fp32[bs]['total'],
    'tp_spec_s':timing_fp32[bs]['tp'],'enc_ms':timing_fp32[bs]['enc'],
    'nar_ms':timing_fp32[bs]['nar'],'p50':timing_fp32[bs]['p50'],
    'p95':timing_fp32[bs]['p95']} for bs in BATCH_SIZES]).round(3)
print(f'\n── Stage breakdown (FP32, bs=1, {b1["n_spec"]} spectra) ──')
print(df_stage_fp32.to_string(index=False))
print(f'\n── Throughput (FP32) ──\n{df_tp_fp32.to_string(index=False)}')
print(f'\nGPU util: {gpu_util_fp32:.0f}%  |  Peak VRAM: {gpu_vram_fp32:.2f} GB')
_tgt = lambda ms: 'MEETS ✓' if ms<=10 else f'FAILS ({ms:.1f}ms)'
print(f'10ms target (bs=1): {_tgt(b1["total"])}')
df_stage_fp32.to_csv('results/fp32_stage_bs1.csv',index=False)
df_tp_fp32.to_csv('results/fp32_throughput.csv',index=False)


══ FP32 Baseline  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=1: 100%|██████████| 5000/5000 [01:47<00:00, 46.44spec/s]


  total=21.13ms  enc=8.14ms  nar=11.21ms  tp=47.9spec/s

══ FP32 Baseline  batch_size=   8 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

...

  bs=128: 5120spec [00:09, 522.29spec/s]                        

  total=1.90ms  enc=0.56ms  nar=1.14ms  tp=525.8spec/s

══ FP32 Baseline  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=512: 5120spec [00:10, 492.73spec/s]                        


  total=2.03ms  enc=0.66ms  nar=1.20ms  tp=493.5spec/s

── Stage breakdown (FP32, bs=1, 5000 spectra) ──
           Stage  mean_ms  p50_ms  p95_ms
DataLoader fetch    1.445   1.392   1.938
    H2D transfer    0.227   0.216   0.314
 SpectrumEncoder    8.139   7.720  11.892
     NAR Decoder   11.207  10.750  14.716
    Output write    0.112   0.104   0.142
           TOTAL   21.130  20.247  28.302

── Throughput (FP32) ──
 batch_size  total_ms  tp_spec_s  enc_ms  nar_ms    p50    p95
          1    21.130     47.929   8.139  11.207 20.247 28.302
          8     3.134    325.919   1.118   1.582  2.903  4.292
         32     1.854    540.112   0.607   0.959  1.849  1.969
        128     1.903    525.819   0.557   1.145  1.895  1.978
        512     2.027    493.526   0.656   1.199  2.027  2.066

GPU util: 33%  |  Peak VRAM: 2.51 GB
10ms target (bs=1): FAILS (21.1ms)


In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — BF16 + FlashAttention Timing
# FIX: the previous version reused _mz/_it/_pr from Cell 1's smoke-test,
# but Cell 2's timing loop reassigns the name `_it` to a DataLoader
# iterator object (`_it = iter(...)`), silently shadowing the original
# intensity tensor. Fetching a fresh batch here avoids depending on any
# global state from earlier cells.
# ═══════════════════════════════════════════════════════════════════════
assert BF16_SUPPORTED, 'BF16 not supported on this GPU.'

# ── Quick flash probe (fetch a fresh real spectrum, don't reuse globals) ─
_probe_dm = DeNovoDataModule(
    lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
    eval_batch_size=1, tokenizer=runner.tokenizer,
    max_charge=MODEL_MAX_CHARGE, n_workers=0,
)
_probe_dm.setup(stage='test', annotated=False)
_probe_batch = next(iter(_probe_dm.predict_dataloader()))
_probe_mz, _probe_it, _probe_pr, _ = model._process_batch(_probe_batch)

_pb_mz, _pb_it, _pb_pr = _probe_mz.to(DEVICE), _probe_it.to(DEVICE), _probe_pr.to(DEVICE)
_pz = torch.zeros((1, model.max_peptide_len), dtype=torch.long, device=DEVICE)
FLASH_ACCEPTED = False
try:
    with torch.no_grad(), _bf16_ctx():
        with sdpa_kernel(backends=[SDPBackend.FLASH_ATTENTION]):
            _pme, _pmk = model.encoder(_pb_mz, _pb_it)
            model.decoder(tokens=_pz, memory=_pme,
                          memory_key_padding_mask=_pmk, precursors=_pb_pr)
    FLASH_ACCEPTED = True
except RuntimeError as _e:
    _flash_err = str(_e)[:120]
_sync()

print('── FlashAttention probe (BF16 + flash_compatible=True) ──────')
if FLASH_ACCEPTED:
    print('  RESULT: Flash ACCEPTED ✓ — FlashAttention now active for decoder self-attention.')
else:
    print(f'  RESULT: Flash rejected — {_flash_err}')
    print('  NOTE: Decoder cross-attention (memory_key_padding_mask present) '
          'will still use memory-efficient attention.')
print('─────────────────────────────────────────────────────────────\n')

# ── Timing ────────────────────────────────────────────────────────────
_gpu_s, _gpu_stop = _start_gpu_monitor()
timing_bf16 = {}

for bs in BATCH_SIZES:
    print(f'\n══ BF16+Flash  batch_size={bs:4d} ══')
    _dm_bs = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                              eval_batch_size=bs, tokenizer=runner.tokenizer,
                              max_charge=MODEL_MAX_CHARGE, n_workers=0)
    _dm_bs.setup(stage='test', annotated=False)
    with torch.no_grad(), _bf16_ctx():
        for _w,_wb in enumerate(iter(_dm_bs.predict_dataloader())):
            if _w>=N_WARMUP_BATCHES: break
            _wm,_wi,_wp,_ = model._process_batch(_wb)
            _wm=_wm.to(DEVICE); _wi=_wi.to(DEVICE); _wp=_wp.to(DEVICE)
            _wme,_wmk = model.encoder(_wm,_wi)
            model.decoder(tokens=torch.zeros((_wm.shape[0],model.max_peptide_len),
                          dtype=torch.long,device=DEVICE),
                          memory=_wme,memory_key_padding_mask=_wmk,precursors=_wp)
    _sync()

    _t={k:[] for k in ['fetch','h2d','enc','nar','write','total','tp']}
    _loader_iter=iter(_dm_bs.predict_dataloader()); n_spec=0
    pbar=tqdm(total=N_TIMING_SPECTRA,desc=f'  bs={bs}',unit='spec')
    while n_spec<N_TIMING_SPECTRA:
        _sync(); t0=time.perf_counter()
        try: batch=next(_loader_iter)
        except StopIteration: _loader_iter=iter(_dm_bs.predict_dataloader()); batch=next(_loader_iter)
        t_fetch=(time.perf_counter()-t0)*1000
        _sync(); t0=time.perf_counter()
        mzs,ints,precs,_=model._process_batch(batch)
        mzs=mzs.to(DEVICE); ints=ints.to(DEVICE); precs=precs.to(DEVICE)
        _sync(); t_h2d=(time.perf_counter()-t0)*1000; ab=mzs.shape[0]
        with torch.no_grad(), _bf16_ctx():
            _sync(); t0=time.perf_counter()
            mem,mmk=model.encoder(mzs,ints)
            _sync(); t_enc=(time.perf_counter()-t0)*1000
            zt=torch.zeros((ab,model.max_peptide_len),dtype=torch.long,device=DEVICE)
            _sync(); t0=time.perf_counter()
            scores=model.decoder(tokens=zt,memory=mem,
                                 memory_key_padding_mask=mmk,precursors=precs)
            _sync(); t_nar=(time.perf_counter()-t0)*1000
        t0=time.perf_counter()
        pred=scores.argmax(dim=-1).cpu(); _=[{'tokens':t.tolist()} for t in pred]
        t_write=(time.perf_counter()-t0)*1000
        tt=t_fetch+t_h2d+t_enc+t_nar+t_write
        for k,v in zip(['fetch','h2d','enc','nar','write','total','tp'],
                       [t_fetch/ab,t_h2d/ab,t_enc/ab,t_nar/ab,t_write/ab,tt/ab,ab/(tt/1000)]):
            _t[k].append(v)
        n_spec+=ab; pbar.update(ab)
        if n_spec>=N_TIMING_SPECTRA: break
    pbar.close()
    p=lambda a,q: float(np.percentile(a,q))
    timing_bf16[bs]={'n_spec':n_spec,'fetch':np.mean(_t['fetch']),'h2d':np.mean(_t['h2d']),
        'enc':np.mean(_t['enc']),'nar':np.mean(_t['nar']),'write':np.mean(_t['write']),
        'total':np.mean(_t['total']),'p50':p(_t['total'],50),'p95':p(_t['total'],95),
        'tp':np.mean(_t['tp']),'raw':_t}
    s=timing_bf16[bs]
    print(f'  total={s["total"]:.2f}ms  enc={s["enc"]:.2f}ms  nar={s["nar"]:.2f}ms  tp={s["tp"]:.1f}spec/s')
    spd=timing_fp32[bs]['total']/max(s['total'],0.001)
    print(f'  vs FP32: {spd:.2f}×')
    if DEVICE=='cuda': torch.cuda.empty_cache()

_gpu_stop.set(); time.sleep(1.0)
gpu_util_bf16=np.mean([s[0] for s in _gpu_s]) if _gpu_s else 0
gpu_vram_bf16=np.max([s[1] for s in _gpu_s]) if _gpu_s else 0

c1=timing_bf16[1]; rc=c1['raw']
df_stage_bf16=pd.DataFrame([
    {'Stage':'DataLoader fetch',   'mean_ms':c1['fetch'],'p50_ms':np.percentile(rc['fetch'],50),'p95_ms':np.percentile(rc['fetch'],95)},
    {'Stage':'H2D transfer',       'mean_ms':c1['h2d'],  'p50_ms':np.percentile(rc['h2d'],50),  'p95_ms':np.percentile(rc['h2d'],95)},
    {'Stage':'SpectrumEncoder(BF16)','mean_ms':c1['enc'],'p50_ms':np.percentile(rc['enc'],50),  'p95_ms':np.percentile(rc['enc'],95)},
    {'Stage':'NAR Decoder(BF16+Flash)','mean_ms':c1['nar'],'p50_ms':np.percentile(rc['nar'],50),'p95_ms':np.percentile(rc['nar'],95)},
    {'Stage':'Output write',       'mean_ms':c1['write'],'p50_ms':np.percentile(rc['write'],50),'p95_ms':np.percentile(rc['write'],95)},
    {'Stage':'TOTAL',              'mean_ms':c1['total'],'p50_ms':c1['p50'],                   'p95_ms':c1['p95']},
]).round(3)
df_tp_bf16=pd.DataFrame([{'batch_size':bs,'total_ms':timing_bf16[bs]['total'],
    'tp_spec_s':timing_bf16[bs]['tp'],'enc_ms':timing_bf16[bs]['enc'],
    'nar_ms':timing_bf16[bs]['nar'],'p50':timing_bf16[bs]['p50'],
    'p95':timing_bf16[bs]['p95'],'vs_fp32':f"{timing_fp32[bs]['total']/max(timing_bf16[bs]['total'],0.001):.2f}x"}
    for bs in BATCH_SIZES]).round(3)
print(f'\n── Stage breakdown (BF16+Flash, bs=1) ──\n{df_stage_bf16.to_string(index=False)}')
print(f'\n── Throughput (BF16+Flash) ──\n{df_tp_bf16.to_string(index=False)}')
print(f'\nGPU util: {gpu_util_bf16:.0f}%  |  Peak VRAM: {gpu_vram_bf16:.2f} GB')
print(f'10ms target (bs=1): {_tgt(c1["total"])}')
_save(df_stage_bf16, 'bf16_stage_bs1.csv')
_save(df_tp_bf16, 'bf16_throughput.csv')

subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

── FlashAttention probe (BF16 + flash_compatible=True) ──────
  RESULT: Flash rejected — No available kernel. Aborting execution.
  NOTE: Decoder cross-attention (memory_key_padding_mask present) will still use memory-efficient attention.
─────────────────────────────────────────────────────────────


══ BF16+Flash  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=1: 100%|██████████| 5000/5000 [02:12<00:00, 37.71spec/s]


  total=25.82ms  enc=9.46ms  nar=14.58ms  tp=39.1spec/s
  vs FP32: 0.82×

══ BF16+Flash  batch_size=   8 ══


...

  total=1.06ms  enc=0.37ms  nar=0.48ms  tp=945.6spec/s
  vs FP32: 1.80×

══ BF16+Flash  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=512: 5120spec [00:05, 952.06spec/s]                        


  total=1.05ms  enc=0.40ms  nar=0.45ms  tp=956.4spec/s
  vs FP32: 1.94×

── Stage breakdown (BF16+Flash, bs=1) ──
                  Stage  mean_ms  p50_ms  p95_ms
       DataLoader fetch    1.424   1.370   1.900
           H2D transfer    0.231   0.220   0.320
  SpectrumEncoder(BF16)    9.458   8.989  13.278
NAR Decoder(BF16+Flash)   14.582  14.090  17.789
           Output write    0.129   0.122   0.163
                  TOTAL   25.824  24.901  33.856

── Throughput (BF16+Flash) ──
 batch_size  total_ms  tp_spec_s  enc_ms  nar_ms    p50    p95 vs_fp32
          1    25.824     39.116   9.458  14.582 24.901 33.856   0.82x
          8     3.908    261.656   1.348   2.117  3.580  5.349   0.80x
         32     1.186    861.785   0.361   0.558  1.083  1.520   1.56x
        128     1.059    945.593   0.365   0.477  1.038  1.133   1.80x
        512     1.046    956.421   0.403   0.450  1.044  1.074   1.94x

GPU util: 21%  |  Peak VRAM: 2.76 GB
10ms target (bs=1): FAILS (25.8ms)
Saved: /teams

'/teamspace/studios/this_studio/profiling_after_depthcharge_changes/results/bf16_throughput.csv'

In [6]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — torch.compile + BF16 + FlashAttention Timing
# Two depthcharge fixes now enable better CUDA Graph capture:
#   (1) spectra.py: new_zeros() instead of torch.tensor([[False]]*n)
#       → eliminates the data-dependent graph break in SpectrumEncoder
#   (2) flash_compatible=True (via patch): no mask tensors in decoder
#       → cleaner decoder graph with no data-dependent tensor injection
# Still requires N_PEAKS=150 fixed shapes for CUDA Graph replay.
# Still requires cudagraph_mark_step_begin() when chaining two
# separately compiled modules (encoder → decoder).
# ═══════════════════════════════════════════════════════════════════════
torch._dynamo.config.cache_size_limit = 32

compiled_encoder = torch.compile(model.encoder, mode='reduce-overhead')
compiled_decoder = torch.compile(model.decoder, mode='reduce-overhead')
print('compiled_encoder / compiled_decoder created (mode=reduce-overhead)')
print('First call at each batch size triggers JIT + CUDA Graph capture.')

_gpu_s, _gpu_stop = _start_gpu_monitor()
timing_compiled = {}

for bs in BATCH_SIZES:
    print(f'\n══ Compiled+BF16+Flash  batch_size={bs:4d} ══')
    _dm_bs = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                              eval_batch_size=bs, tokenizer=runner.model.tokenizer,
                              max_charge=MODEL_MAX_CHARGE, n_workers=0)
    _dm_bs.setup(stage='test', annotated=False)

    print(f'  Compiling + capturing CUDA Graph for bs={bs}…')
    _t0_compile = time.perf_counter()
    with torch.no_grad(), _bf16_ctx():
        for _w,_wb in enumerate(iter(_dm_bs.predict_dataloader())):
            if _w>=N_WARMUP_BATCHES: break
            _wm,_wi,_wp,_ = model._process_batch(_wb)
            _wm=_wm.to(DEVICE); _wi=_wi.to(DEVICE); _wp=_wp.to(DEVICE)
            _wm,_wi,_ = pad_or_select_to_fixed_peaks(_wm,_wi)
            _mark_step()
            _wme,_wmk = compiled_encoder(_wm,_wi)
            compiled_decoder(tokens=torch.zeros((_wm.shape[0],model.max_peptide_len),
                             dtype=torch.long,device=DEVICE),
                             memory=_wme,memory_key_padding_mask=_wmk,precursors=_wp)
    _sync()
    compile_s = time.perf_counter()-_t0_compile
    print(f'  Compile+capture done in {compile_s:.1f}s (excluded from timings below)')

    _t={k:[] for k in ['fetch','h2d','enc','nar','write','total','tp']}
    _it=iter(_dm_bs.predict_dataloader()); n_spec=0
    pbar=tqdm(total=N_TIMING_SPECTRA,desc=f'  bs={bs}',unit='spec')
    while n_spec<N_TIMING_SPECTRA:
        _sync(); t0=time.perf_counter()
        try: batch=next(_it)
        except StopIteration: _it=iter(_dm_bs.predict_dataloader()); batch=next(_it)
        t_fetch=(time.perf_counter()-t0)*1000
        _sync(); t0=time.perf_counter()
        mzs,ints,precs,_=model._process_batch(batch)
        mzs=mzs.to(DEVICE); ints=ints.to(DEVICE); precs=precs.to(DEVICE)
        mzs,ints,_ntrunc=pad_or_select_to_fixed_peaks(mzs,ints)
        _sync(); t_h2d=(time.perf_counter()-t0)*1000; ab=mzs.shape[0]
        with torch.no_grad(), _bf16_ctx():
            _mark_step()
            _sync(); t0=time.perf_counter()
            mem,mmk=compiled_encoder(mzs,ints)
            _sync(); t_enc=(time.perf_counter()-t0)*1000
            zt=torch.zeros((ab,model.max_peptide_len),dtype=torch.long,device=DEVICE)
            _sync(); t0=time.perf_counter()
            scores=compiled_decoder(tokens=zt,memory=mem,
                                    memory_key_padding_mask=mmk,precursors=precs)
            _sync(); t_nar=(time.perf_counter()-t0)*1000
        t0=time.perf_counter()
        pred=scores.argmax(dim=-1).cpu(); _=[{'tokens':t.tolist()} for t in pred]
        t_write=(time.perf_counter()-t0)*1000
        tt=t_fetch+t_h2d+t_enc+t_nar+t_write
        for k,v in zip(['fetch','h2d','enc','nar','write','total','tp'],
                       [t_fetch/ab,t_h2d/ab,t_enc/ab,t_nar/ab,t_write/ab,tt/ab,ab/(tt/1000)]):
            _t[k].append(v)
        n_spec+=ab; pbar.update(ab)
        if n_spec>=N_TIMING_SPECTRA: break
    pbar.close()
    p=lambda a,q:float(np.percentile(a,q))
    timing_compiled[bs]={'n_spec':n_spec,'compile_s':compile_s,
        'fetch':np.mean(_t['fetch']),'h2d':np.mean(_t['h2d']),
        'enc':np.mean(_t['enc']),'nar':np.mean(_t['nar']),'write':np.mean(_t['write']),
        'total':np.mean(_t['total']),'p50':p(_t['total'],50),'p95':p(_t['total'],95),
        'tp':np.mean(_t['tp']),'raw':_t}
    s=timing_compiled[bs]
    spd=timing_fp32[bs]['total']/max(s['total'],0.001)
    print(f'  total={s["total"]:.2f}ms  enc={s["enc"]:.2f}ms  nar={s["nar"]:.2f}ms  '
          f'tp={s["tp"]:.1f}spec/s  vs FP32: {spd:.2f}×')
    if DEVICE=='cuda': torch.cuda.empty_cache()

_gpu_stop.set(); time.sleep(1.0)
gpu_util_comp=np.mean([s[0] for s in _gpu_s]) if _gpu_s else 0
gpu_vram_comp=np.max([s[1] for s in _gpu_s]) if _gpu_s else 0

e1=timing_compiled[1]; re=e1['raw']
df_stage_comp=pd.DataFrame([
    {'Stage':'DataLoader fetch',       'mean_ms':e1['fetch'],'p50_ms':np.percentile(re['fetch'],50),'p95_ms':np.percentile(re['fetch'],95)},
    {'Stage':'H2D+fixed-peak pad',     'mean_ms':e1['h2d'],  'p50_ms':np.percentile(re['h2d'],50),  'p95_ms':np.percentile(re['h2d'],95)},
    {'Stage':'Encoder(Compiled+BF16)', 'mean_ms':e1['enc'],  'p50_ms':np.percentile(re['enc'],50),  'p95_ms':np.percentile(re['enc'],95)},
    {'Stage':'Decoder(Compiled+Flash)','mean_ms':e1['nar'],  'p50_ms':np.percentile(re['nar'],50),  'p95_ms':np.percentile(re['nar'],95)},
    {'Stage':'Output write',           'mean_ms':e1['write'],'p50_ms':np.percentile(re['write'],50),'p95_ms':np.percentile(re['write'],95)},
    {'Stage':'TOTAL',                  'mean_ms':e1['total'],'p50_ms':e1['p50'],                   'p95_ms':e1['p95']},
]).round(3)
df_tp_comp=pd.DataFrame([{'batch_size':bs,'total_ms':timing_compiled[bs]['total'],
    'tp_spec_s':timing_compiled[bs]['tp'],'enc_ms':timing_compiled[bs]['enc'],
    'nar_ms':timing_compiled[bs]['nar'],'compile_s':round(timing_compiled[bs]['compile_s'],1),
    'vs_fp32':f"{timing_fp32[bs]['total']/max(timing_compiled[bs]['total'],0.001):.2f}x"}
    for bs in BATCH_SIZES]).round(3)
print(f'\n── Stage breakdown (Compiled+BF16+Flash, bs=1) ──\n{df_stage_comp.to_string(index=False)}')
print(f'\n── Throughput (Compiled+BF16+Flash) ──\n{df_tp_comp.to_string(index=False)}')
print(f'\nGPU util: {gpu_util_comp:.0f}%  |  Peak VRAM: {gpu_vram_comp:.2f} GB')
print(f'10ms target (bs=1): {_tgt(e1["total"])}')
df_stage_comp.to_csv('results/compiled_stage_bs1.csv',index=False)
df_tp_comp.to_csv('results/compiled_throughput.csv',index=False)

compiled_encoder / compiled_decoder created (mode=reduce-overhead)
First call at each batch size triggers JIT + CUDA Graph capture.

══ Compiled+BF16+Flash  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Compiling + capturing CUDA Graph for bs=1…


W0706 17:08:44.613000 21179 /system/conda/miniconda3/envs/cloudspace/lib/python3.10/site-packages/torch/_inductor/utils.py:1250] [0/0_1] Not enough SMs to use max_autotune_gemm mode


  Compile+capture done in 22.4s (excluded from timings below)


...

  total=0.98ms  enc=0.35ms  nar=0.42ms  tp=1019.2spec/s  vs FP32: 1.94×

══ Compiled+BF16+Flash  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Compiling + capturing CUDA Graph for bs=512…
  Compile+capture done in 5.3s (excluded from timings below)


  bs=512: 5120spec [00:04, 1051.35spec/s]                        


  total=0.95ms  enc=0.38ms  nar=0.34ms  tp=1057.1spec/s  vs FP32: 2.14×

── Stage breakdown (Compiled+BF16+Flash, bs=1) ──
                  Stage  mean_ms  p50_ms  p95_ms
       DataLoader fetch    1.311   1.254   1.783
     H2D+fixed-peak pad    0.304   0.305   0.440
 Encoder(Compiled+BF16)    9.489   9.051  13.197
Decoder(Compiled+Flash)    1.838   1.813   2.054
           Output write    0.132   0.126   0.173
                  TOTAL   13.074  12.549  17.496

── Throughput (Compiled+BF16+Flash) ──
 batch_size  total_ms  tp_spec_s  enc_ms  nar_ms  compile_s vs_fp32
          1    13.074     77.454   9.489   1.838       22.4   1.62x
          8     2.526    399.845   1.313   0.767       33.6   1.24x
         32     1.026    989.500   0.389   0.342        1.6   1.81x
        128     0.983   1019.179   0.351   0.422        2.6   1.94x
        512     0.947   1057.050   0.383   0.340        5.3   2.14x

GPU util: 22%  |  Peak VRAM: 3.99 GB
10ms target (bs=1): FAILS (13.1ms)


In [7]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — torch.profiler: FP32 vs BF16+Flash vs Compiled+BF16+Flash
# Key diagnostics:
#   (1) Attention kernel: flash vs efficient (per variant)
#   (2) Torch-Compiled Regions: how many? (should be fewer now — spectra
#       encoder graph-break fix + no mask injection in decoder)
#   (3) Kernel launches per spectrum
# ═══════════════════════════════════════════════════════════════════════
ACTS = ([ProfilerActivity.CPU, ProfilerActivity.CUDA]
        if DEVICE == 'cuda' else [ProfilerActivity.CPU])
N_PROF = PROF_WARMUP + PROF_ACTIVE

print(f'Pre-fetching {N_PROF} bs=1 batches…')
_dm_prof = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                            eval_batch_size=1, tokenizer=runner.model.tokenizer,
                            max_charge=MODEL_MAX_CHARGE, n_workers=0)
_dm_prof.setup(stage='test', annotated=False)

_prof_batches, _prof_batches_fixed = [], []
for _b in _dm_prof.predict_dataloader():
    _mz2, _it2, _pr2, _ = model._process_batch(_b)
    _prof_batches.append((_mz2.to(DEVICE), _it2.to(DEVICE), _pr2.to(DEVICE)))
    _mzf, _itf, _ = pad_or_select_to_fixed_peaks(_mz2.to(DEVICE), _it2.to(DEVICE))
    _prof_batches_fixed.append((_mzf, _itf, _pr2.to(DEVICE)))
    if len(_prof_batches) >= N_PROF: break
while len(_prof_batches) < N_PROF:
    _prof_batches.extend(_prof_batches[:N_PROF-len(_prof_batches)])
    _prof_batches_fixed.extend(_prof_batches_fixed[:N_PROF-len(_prof_batches_fixed)])
print(f'Using {len(_prof_batches)} batches')

_zt_prof = torch.zeros((1, model.max_peptide_len), dtype=torch.long, device=DEVICE)

def _run_prof(label, trace_path, txt_path, batches, ctx_fn, enc_fn, dec_fn,
              use_mark_step=False, n_warm=10):
    with torch.no_grad(), ctx_fn():
        for _mz2,_it2,_pr2 in batches[:n_warm]:
            if use_mark_step: _mark_step()
            _me,_mk = enc_fn(_mz2,_it2)
            dec_fn(tokens=_zt_prof,memory=_me,memory_key_padding_mask=_mk,precursors=_pr2)
    _sync()
    _store={}
    def _on_ready(p):
        p.export_chrome_trace(trace_path)
        _store['tbl']  = p.key_averages().table(sort_by='cpu_time_total', row_limit=12)
        _store['avgs'] = p.key_averages()
    with profile(activities=ACTS, record_shapes=True,
                 schedule=schedule(wait=0, warmup=PROF_WARMUP, active=PROF_ACTIVE),
                 on_trace_ready=_on_ready) as p:
        with torch.no_grad(), ctx_fn():
            for _mz2,_it2,_pr2 in batches:
                if use_mark_step: _mark_step()
                with record_function(label):
                    _me,_mk = enc_fn(_mz2,_it2)
                    dec_fn(tokens=_zt_prof,memory=_me,
                           memory_key_padding_mask=_mk,precursors=_pr2)
                _sync(); p.step()
    print(_store.get('tbl','(no data)'))
    with open(txt_path,'w') as f:
        f.write(f'{label}  bs=1  warmup={PROF_WARMUP}  active={PROF_ACTIVE}\n')
        f.write('='*64+'\n'+str(_store.get('tbl','no data')))
    print(f'Chrome trace → {trace_path}')
    if DEVICE=='cuda': torch.cuda.synchronize(); torch.cuda.empty_cache()
    return _store

@contextmanager
def _fp32_ctx():
    yield   # plain eager FP32 — no autocast

# ── A) Baseline FP32 ──────────────────────────────────────────────────
print('\n── A) torch.profiler: Baseline FP32 (eager) ─────────────────')
_store_a = _run_prof('fp32_full_forward', 'results/trace_fp32.json',
                     'results/profiler_fp32.txt', _prof_batches, _fp32_ctx,
                     model.encoder, model.decoder)
_attn_a = _detect_attention_kernel(_store_a, 'A) FP32')
_nkern_a = _launch_count(_store_a)
_nreg_a  = _compiled_region_count(_store_a)

# ── B) BF16 + Flash ───────────────────────────────────────────────────
print('\n── B) torch.profiler: BF16 + FlashAttention (eager) ─────────')
_store_b = _run_prof('bf16_flash_forward', 'results/trace_bf16.json',
                     'results/profiler_bf16.txt', _prof_batches, _bf16_ctx,
                     model.encoder, model.decoder)
_attn_b = _detect_attention_kernel(_store_b, 'B) BF16+Flash')
_nkern_b = _launch_count(_store_b)
_nreg_b  = _compiled_region_count(_store_b)

# ── C) Compiled + BF16 + Flash ────────────────────────────────────────
print('\n── C) torch.profiler: Compiled + BF16 + Flash ───────────────')
_store_c = _run_prof('compiled_bf16_flash', 'results/trace_compiled.json',
                     'results/profiler_compiled.txt', _prof_batches_fixed, _bf16_ctx,
                     compiled_encoder, compiled_decoder, use_mark_step=True)
_attn_c = _detect_attention_kernel(_store_c, 'C) Compiled+BF16+Flash')
_nkern_c = _launch_count(_store_c)
_nreg_c  = _compiled_region_count(_store_c)

print('\n── Profiler Comparison Summary ───────────────────────────────')
print(f'{"Metric":<35} {"A:FP32":>12} {"B:BF16+Flash":>14} {"C:Compiled":>12}')
print('-'*76)
print(f'{"Kernel launches/spectrum":<35} {str(_nkern_a):>12} {str(_nkern_b):>14} {str(_nkern_c):>12}')
print(f'{"Torch-Compiled Regions":<35} {str(_nreg_a or "N/A"):>12} {str(_nreg_b or "N/A"):>14} {str(_nreg_c or "N/A"):>12}')
print(f'{"Attention kernel":<35} {"efficient":>12} '
      f'{"flash✓" if FLASH_ACCEPTED else "efficient":>14} '
      f'{"flash✓" if FLASH_ACCEPTED else "efficient":>12}')

Pre-fetching 70 bs=1 batches…


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

Using 70 batches

── A) torch.profiler: Baseline FP32 (eager) ─────────────────
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.34%       7.611ms       100.00%        2.226s      44.512ms       0.000us         0.00%     147.424ms       2.948ms            50  
                                      fp32_full_forward        22.90%     509.640ms        99.58%        2.216s      44.324ms  

In [8]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — Comparison Plots: FP32 vs BF16+Flash vs Compiled+BF16+Flash
# ═══════════════════════════════════════════════════════════════════════
COLORS = {'fp32': '#D85A30', 'bf16': '#1D9E75', 'compiled': '#3A7FC1'}
LABELS = {
    'fp32'    : 'Baseline FP32 (eager)',
    'bf16'    : 'BF16 + Flash (eager)',
    'compiled': 'Compiled + BF16 + Flash',
}
_xi   = list(range(len(BATCH_SIZES)))
_xlbl = [str(b) for b in BATCH_SIZES]
_tps  = {k: [timing[bs]['tp']    for bs in BATCH_SIZES]
         for k, timing in [('fp32',timing_fp32),('bf16',timing_bf16),('compiled',timing_compiled)]}
_lats = {k: [timing[bs]['total'] for bs in BATCH_SIZES]
         for k, timing in [('fp32',timing_fp32),('bf16',timing_bf16),('compiled',timing_compiled)]}

# ── Figure 1 — Stage breakdown bs=1 ──────────────────────────────────
fig1, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
fig1.suptitle('Stage Breakdown (bs=1) — Three Variants', fontweight='bold')
for ax, (key, df, total) in zip(axes, [
    ('fp32',     df_stage_fp32, timing_fp32[1]['total']),
    ('bf16',     df_stage_bf16, timing_bf16[1]['total']),
    ('compiled', df_stage_comp, timing_compiled[1]['total']),
]):
    _stages = ['Fetch','H2D','Encoder','Decoder','Write']
    _vals   = df[df['Stage'] != 'TOTAL']['mean_ms'].values
    bars = ax.bar(_stages, _vals, color=COLORS[key], edgecolor='none', width=0.55)
    for b, v in zip(bars, _vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.15, f'{v:.2f}',
                ha='center', fontsize=8)
    ax.set_title(f'{LABELS[key]}\nTotal: {total:.1f} ms/spec')
    ax.set_ylabel('ms / spectrum')
    ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/stage_comparison.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: results/stage_comparison.png')

# ── Figure 2 — Throughput + Latency vs batch size ────────────────────
fig2, (ax_tp, ax_lat) = plt.subplots(1, 2, figsize=(14, 5))
fig2.suptitle('NAR Performance vs Batch Size — Three Variants', fontweight='bold')
for key in ['fp32','bf16','compiled']:
    ax_tp.plot(_xi, _tps[key], 'o-', color=COLORS[key], lw=2, ms=8, label=LABELS[key])
    ax_lat.plot(_xi, _lats[key], 'o-', color=COLORS[key], lw=2, ms=8, label=LABELS[key])
for ax, ylabel, title in [
    (ax_tp, 'Throughput (spec/s)', 'Throughput vs Batch Size'),
    (ax_lat, 'ms / spectrum',      'Latency vs Batch Size'),
]:
    ax.set_xticks(_xi); ax.set_xticklabels(_xlbl)
    ax.set_xlabel('Batch size'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=8, frameon=False); ax.spines[['top','right']].set_visible(False)
ax_lat.axhline(10, color='black', lw=1.5, ls=':', label='10ms (100Hz)')
ax_lat.axhline(35, color='purple', lw=1.2, ls='--', label='35ms (29Hz)')
ax_lat.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.savefig('results/throughput_comparison.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: results/throughput_comparison.png')

# ── Figure 3 — Latency distribution histograms (bs=1) ────────────────
fig3, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
fig3.suptitle('Latency Distribution (bs=1, 5000 spectra) — Three Variants', fontweight='bold')
for ax, (key, timing) in zip(axes, [('fp32',timing_fp32),('bf16',timing_bf16),('compiled',timing_compiled)]):
    _raw = timing[1]['raw']['total']
    ax.hist(_raw, bins=30, color=COLORS[key], alpha=0.8, edgecolor='none')
    ax.axvline(np.mean(_raw), color='black', lw=2, ls='--',
               label=f'mean={np.mean(_raw):.1f}ms')
    ax.axvline(10, color='red', lw=1.5, ls=':', label='10ms target')
    ax.set_title(LABELS[key]); ax.set_xlabel('ms / spectrum')
    ax.legend(fontsize=8, frameon=False); ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('results/latency_histograms.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved: results/latency_histograms.png')

Saved: results/stage_comparison.png
Saved: results/throughput_comparison.png
Saved: results/latency_histograms.png


In [9]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 — Summary Report
# ═══════════════════════════════════════════════════════════════════════
_now  = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
_spd  = lambda a, b: a['total'] / max(b['total'], 0.001)

summary = f"""CASANOVO NAR PROFILING — Three-Way Comparison
Generated  : {_now}
Hardware   : {GPU_NAME} | {TOTAL_VRAM:.1f} GB VRAM | PyTorch {torch.__version__}
depthcharge: {_dc.__file__}  (local branch — nar-flash-cuda-graph-compat)

DEPTHCHARGE CHANGES IN THIS BRANCH
  analytes.py : AnalyteTransformerDecoder.embed() + forward()
    — New param flash_compatible=False (fully backward-compatible)
    — When True: suppresses causal tgt_mask + tgt_key_padding_mask
    — Enables PyTorch FlashAttention SDPA backend for decoder self-attention
  spectra.py  : SpectrumTransformerEncoder.forward()
    — Replaced torch.tensor([[False]]*batch_size) with new_zeros()
    — Eliminates data-dependent Python control flow that caused
      TorchDynamo graph breaks in CUDA Graph capture (was 4 fragments)

NAR PATCH (Cell 0) CHANGE
  Old: injected torch.zeros((L,L), dtype=torch.bool) as tgt_mask
  New: passes flash_compatible=True → parent handles everything cleanly

FLASHATTENTION PROBE RESULT
  flash_compatible=True + BF16 : {'ACCEPTED ✓' if FLASH_ACCEPTED else 'REJECTED — see profiler for actual kernel used'}
  {_attn_a}
  {_attn_b}
  {_attn_c}

KERNEL LAUNCH COMPARISON (bs=1, 50 profiled spectra)
  FP32 baseline          : {_nkern_a} launches/spectrum
  BF16 + Flash           : {_nkern_b} launches/spectrum
  Compiled + BF16 + Flash: {_nkern_c} launches/spectrum

TORCH-COMPILED REGIONS (bs=1 profiler)
  FP32 baseline          : {_nreg_a or 'N/A'} regions (eager — expected N/A)
  BF16 + Flash           : {_nreg_b or 'N/A'} regions (eager — expected N/A)
  Compiled + BF16 + Flash: {_nreg_c} regions  (was 4 before spectra.py fix; expect 1-2 now)

TIMING RESULTS (real spectra, 5000 per batch size)
{"Batch":>6} {"FP32(ms)":>10} {"BF16+Flash(ms)":>16} {"Compiled(ms)":>14} {"BF16 vs FP32":>14} {"Comp vs FP32":>14}
{"-"*76}
"""
for bs in BATCH_SIZES:
    f=timing_fp32[bs]['total']; b=timing_bf16[bs]['total']; c=timing_compiled[bs]['total']
    summary += f'{bs:>6} {f:>10.2f} {b:>16.2f} {c:>14.2f} {f/max(b,0.001):>13.2f}× {f/max(c,0.001):>13.2f}×\n'

summary += f"""
STAGE BREAKDOWN (bs=1, 5000 spectra)
FP32 Baseline:
{df_stage_fp32.to_string(index=False)}

BF16 + Flash:
{df_stage_bf16.to_string(index=False)}

Compiled + BF16 + Flash:
{df_stage_comp.to_string(index=False)}

GPU UTILIZATION
  FP32 baseline          : {gpu_util_fp32:.0f}% util | {gpu_vram_fp32:.2f} GB VRAM
  BF16 + Flash           : {gpu_util_bf16:.0f}% util | {gpu_vram_bf16:.2f} GB VRAM
  Compiled + BF16 + Flash: {gpu_util_comp:.0f}% util | {gpu_vram_comp:.2f} GB VRAM

COMPILE COST (one-time, excluded from timings)
{df_tp_comp[['batch_size','compile_s']].to_string(index=False)}

ARTIFACTS
  results/stage_comparison.png
  results/throughput_comparison.png
  results/latency_histograms.png
  results/trace_fp32.json | results/trace_bf16.json | results/trace_compiled.json
  results/profiler_fp32.txt | results/profiler_bf16.txt | results/profiler_compiled.txt
  results/fp32_stage_bs1.csv | results/bf16_stage_bs1.csv | results/compiled_stage_bs1.csv
  results/fp32_throughput.csv | results/bf16_throughput.csv | results/compiled_throughput.csv
  results/nar_summary.txt
"""
print(summary)
with open('results/nar_summary.txt', 'w') as f: f.write(summary)

print('\n── results/ ──')
for _f in sorted(os.listdir('results')):
    _fp = os.path.join('results', _f)
    print(f'  {_f:<50} {os.path.getsize(_fp)/1024:.1f} KB')
print('\nProfiling complete.')

CASANOVO NAR PROFILING — Three-Way Comparison
Generated  : 2026-07-06 17:14
Hardware   : NVIDIA L4 | 23.6 GB VRAM | PyTorch 2.7.1+cu128
depthcharge: /teamspace/studios/this_studio/depthcharge_changes/depthcharge/__init__.py  (local branch — nar-flash-cuda-graph-compat)

DEPTHCHARGE CHANGES IN THIS BRANCH
  analytes.py : AnalyteTransformerDecoder.embed() + forward()
    — New param flash_compatible=False (fully backward-compatible)
    — When True: suppresses causal tgt_mask + tgt_key_padding_mask
    — Enables PyTorch FlashAttention SDPA backend for decoder self-attention
  spectra.py  : SpectrumTransformerEncoder.forward()
    — Replaced torch.tensor([[False]]*batch_size) with new_zeros()
    — Eliminates data-dependent Python control flow that caused
      TorchDynamo graph breaks in CUDA Graph capture (was 4 fragments)

NAR PATCH (Cell 0) CHANGE
  Old: injected torch.zeros((L,L), dtype=torch.bool) as tgt_mask
  New: passes flash_compatible=True → parent handles everything cleanly

F

In [10]:
# ═══════════════════════════════════════════════════════════════════════
# TF32 CELL 1 — Enable TF32 globally + verify environment
# TF32 is a GLOBAL runtime flag affecting all FP32 matmul/conv ops on
# Ampere+ GPUs (L4 qualifies — compute capability 8.9). It does NOT
# change tensor dtype (tensors stay torch.float32) and needs NO autocast
# context — this is the key difference from BF16, and why it avoids the
# casting overhead (aten::to / aten::copy_) that hurt bs=1 performance.
# ═══════════════════════════════════════════════════════════════════════
import torch

assert 'model' in dir(), 'model not found — run Cells 0-9 first in this session.'
assert 'runner' in dir(), 'runner not found — run Cells 0-9 first in this session.'

_tf32_before = (torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print('── TF32 Configuration ──────────────────────────────────────')
print(f'  Before : matmul.allow_tf32={_tf32_before[0]}, cudnn.allow_tf32={_tf32_before[1]}')
print(f'  After  : matmul.allow_tf32={torch.backends.cuda.matmul.allow_tf32}, '
      f'cudnn.allow_tf32={torch.backends.cudnn.allow_tf32}')
print(f'  GPU compute capability: {torch.cuda.get_device_capability(0)} '
      f'(TF32 requires ≥(8,0) — {"supported ✓" if torch.cuda.get_device_capability(0) >= (8,0) else "NOT supported ✗"})')
print('  Note: this flag persists for the rest of this Python process.')
print('  No autocast context needed — tensors remain torch.float32 throughout.')
print('──────────────────────────────────────────────────────────────')

# Sanity check: confirm patch + model still intact from earlier session
from casanovo.denovo.transformers import PeptideDecoder
assert PeptideDecoder.embed is _nar_embed, 'NAR patch lost — re-run Cell 0 (earlier session)!'
print('\nNAR patch still active ✓ | Ready to profile FP32+TF32')

── TF32 Configuration ──────────────────────────────────────
  Before : matmul.allow_tf32=False, cudnn.allow_tf32=True
  After  : matmul.allow_tf32=True, cudnn.allow_tf32=True
  GPU compute capability: (8, 9) (TF32 requires ≥(8,0) — supported ✓)
  Note: this flag persists for the rest of this Python process.
  No autocast context needed — tensors remain torch.float32 throughout.
──────────────────────────────────────────────────────────────

NAR patch still active ✓ | Ready to profile FP32+TF32


In [11]:
# ═══════════════════════════════════════════════════════════════════════
# TF32 CELL 2 — FP32+TF32 Timing (eager mode, no compile)
# Identical loop structure to the original FP32 baseline (Cell 4) — the
# ONLY difference is the TF32 flags enabled in TF32 Cell 1. No autocast,
# no dtype casting, no code path change — pure eager FP32 with TF32
# matmul acceleration under the hood.
# ═══════════════════════════════════════════════════════════════════════
_gpu_s, _gpu_stop = _start_gpu_monitor()
timing_tf32 = {}

for bs in BATCH_SIZES:
    print(f'\n══ FP32+TF32  batch_size={bs:4d} ══')
    _dm_bs = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                              eval_batch_size=bs, tokenizer=runner.tokenizer,
                              max_charge=MODEL_MAX_CHARGE, n_workers=0)
    _dm_bs.setup(stage='test', annotated=False)
    with torch.no_grad():
        for _w, _wb in enumerate(iter(_dm_bs.predict_dataloader())):
            if _w >= N_WARMUP_BATCHES: break
            _wm, _wi, _wp, _ = model._process_batch(_wb)
            _wm=_wm.to(DEVICE); _wi=_wi.to(DEVICE); _wp=_wp.to(DEVICE)
            _wme, _wmk = model.encoder(_wm, _wi)
            model.decoder(tokens=torch.zeros((_wm.shape[0], model.max_peptide_len),
                          dtype=torch.long, device=DEVICE),
                          memory=_wme, memory_key_padding_mask=_wmk, precursors=_wp)
    _sync()

    _t = {k: [] for k in ['fetch','h2d','enc','nar','write','total','tp']}
    _loader_iter = iter(_dm_bs.predict_dataloader()); n_spec = 0
    pbar = tqdm(total=N_TIMING_SPECTRA, desc=f'  bs={bs}', unit='spec')
    while n_spec < N_TIMING_SPECTRA:
        _sync(); t0 = time.perf_counter()
        try: batch = next(_loader_iter)
        except StopIteration: _loader_iter = iter(_dm_bs.predict_dataloader()); batch = next(_loader_iter)
        t_fetch = (time.perf_counter()-t0)*1000

        _sync(); t0 = time.perf_counter()
        mzs, ints, precs, _ = model._process_batch(batch)
        mzs=mzs.to(DEVICE); ints=ints.to(DEVICE); precs=precs.to(DEVICE)
        _sync(); t_h2d = (time.perf_counter()-t0)*1000; ab = mzs.shape[0]

        with torch.no_grad():
            _sync(); t0 = time.perf_counter()
            mem, mmk = model.encoder(mzs, ints)
            _sync(); t_enc = (time.perf_counter()-t0)*1000
            zt = torch.zeros((ab, model.max_peptide_len), dtype=torch.long, device=DEVICE)
            _sync(); t0 = time.perf_counter()
            scores = model.decoder(tokens=zt, memory=mem,
                                   memory_key_padding_mask=mmk, precursors=precs)
            _sync(); t_nar = (time.perf_counter()-t0)*1000

        t0 = time.perf_counter()
        pred = scores.argmax(dim=-1).cpu(); _ = [{'tokens': t.tolist()} for t in pred]
        t_write = (time.perf_counter()-t0)*1000

        tt = t_fetch+t_h2d+t_enc+t_nar+t_write
        for k,v in zip(['fetch','h2d','enc','nar','write','total','tp'],
                       [t_fetch/ab,t_h2d/ab,t_enc/ab,t_nar/ab,t_write/ab,tt/ab,ab/(tt/1000)]):
            _t[k].append(v)
        n_spec += ab; pbar.update(ab)
        if n_spec >= N_TIMING_SPECTRA: break
    pbar.close()

    p = lambda a,q: float(np.percentile(a,q))
    timing_tf32[bs] = {'n_spec':n_spec,'fetch':np.mean(_t['fetch']),'h2d':np.mean(_t['h2d']),
        'enc':np.mean(_t['enc']),'nar':np.mean(_t['nar']),'write':np.mean(_t['write']),
        'total':np.mean(_t['total']),'p50':p(_t['total'],50),'p95':p(_t['total'],95),
        'tp':np.mean(_t['tp']),'raw':_t}
    s = timing_tf32[bs]
    spd = timing_fp32[bs]['total'] / max(s['total'], 0.001)
    print(f'  total={s["total"]:.2f}ms  enc={s["enc"]:.2f}ms  nar={s["nar"]:.2f}ms  '
          f'tp={s["tp"]:.1f}spec/s  vs FP32(no TF32): {spd:.2f}×')
    if DEVICE == 'cuda': torch.cuda.empty_cache()

_gpu_stop.set(); time.sleep(1.0)
gpu_util_tf32 = np.mean([s[0] for s in _gpu_s]) if _gpu_s else 0
gpu_vram_tf32 = np.max([s[1] for s in _gpu_s]) if _gpu_s else 0

t1 = timing_tf32[1]; rt = t1['raw']
df_stage_tf32 = pd.DataFrame([
    {'Stage':'DataLoader fetch','mean_ms':t1['fetch'],'p50_ms':np.percentile(rt['fetch'],50),'p95_ms':np.percentile(rt['fetch'],95)},
    {'Stage':'H2D transfer',    'mean_ms':t1['h2d'],  'p50_ms':np.percentile(rt['h2d'],50),  'p95_ms':np.percentile(rt['h2d'],95)},
    {'Stage':'SpectrumEncoder(TF32)','mean_ms':t1['enc'],'p50_ms':np.percentile(rt['enc'],50),'p95_ms':np.percentile(rt['enc'],95)},
    {'Stage':'NAR Decoder(TF32)','mean_ms':t1['nar'],  'p50_ms':np.percentile(rt['nar'],50),  'p95_ms':np.percentile(rt['nar'],95)},
    {'Stage':'Output write',    'mean_ms':t1['write'],'p50_ms':np.percentile(rt['write'],50),'p95_ms':np.percentile(rt['write'],95)},
    {'Stage':'TOTAL',           'mean_ms':t1['total'],'p50_ms':t1['p50'],                   'p95_ms':t1['p95']},
]).round(3)
df_tp_tf32 = pd.DataFrame([{'batch_size':bs,'total_ms':timing_tf32[bs]['total'],
    'tp_spec_s':timing_tf32[bs]['tp'],'enc_ms':timing_tf32[bs]['enc'],
    'nar_ms':timing_tf32[bs]['nar'],'p50':timing_tf32[bs]['p50'],'p95':timing_tf32[bs]['p95'],
    'vs_fp32':f"{timing_fp32[bs]['total']/max(timing_tf32[bs]['total'],0.001):.2f}x"}
    for bs in BATCH_SIZES]).round(3)

print(f'\n── Stage breakdown (FP32+TF32, bs=1) ──\n{df_stage_tf32.to_string(index=False)}')
print(f'\n── Throughput (FP32+TF32) ──\n{df_tp_tf32.to_string(index=False)}')
print(f'\nGPU util: {gpu_util_tf32:.0f}%  |  Peak VRAM: {gpu_vram_tf32:.2f} GB')
print(f'10ms target (bs=1): {_tgt(t1["total"])}')
_save(df_stage_tf32, 'tf32_stage_bs1.csv')
_save(df_tp_tf32, 'tf32_throughput.csv')


══ FP32+TF32  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=1: 100%|██████████| 5000/5000 [02:11<00:00, 37.94spec/s]


  total=25.80ms  enc=9.99ms  nar=13.80ms  tp=39.2spec/s  vs FP32(no TF32): 0.82×

══ FP32+TF32  batch_size=   8 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

...

  total=1.14ms  enc=0.26ms  nar=0.63ms  tp=879.4spec/s  vs FP32(no TF32): 1.67×

══ FP32+TF32  batch_size= 512 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  bs=512: 5120spec [00:06, 823.86spec/s]                        


  total=1.21ms  enc=0.31ms  nar=0.69ms  tp=826.8spec/s  vs FP32(no TF32): 1.67×

── Stage breakdown (FP32+TF32, bs=1) ──
                Stage  mean_ms  p50_ms  p95_ms
     DataLoader fetch    1.619   1.577   2.119
         H2D transfer    0.266   0.253   0.375
SpectrumEncoder(TF32)    9.987   9.490  14.081
    NAR Decoder(TF32)   13.799  13.304  16.766
         Output write    0.130   0.125   0.167
                TOTAL   25.802  24.835  33.879

── Throughput (FP32+TF32) ──
 batch_size  total_ms  tp_spec_s  enc_ms  nar_ms    p50    p95 vs_fp32
          1    25.802     39.171   9.987  13.799 24.835 33.879   0.82x
          8     3.785    268.922   1.346   1.951  3.546  5.199   0.83x
         32     1.234    828.987   0.386   0.534  1.149  1.569   1.50x
        128     1.139    879.408   0.260   0.633  1.129  1.232   1.67x
        512     1.210    826.849   0.308   0.688  1.200  1.266   1.67x

GPU util: 18%  |  Peak VRAM: 4.51 GB
10ms target (bs=1): FAILS (25.8ms)
Saved: /teamspace/stu

'/teamspace/studios/this_studio/profiling_after_depthcharge_changes/results/tf32_throughput.csv'

In [12]:
# ═══════════════════════════════════════════════════════════════════════
# TF32 CELL 3 — Does TF32 enable FlashAttention? (empirical, honest test)
# IMPORTANT CAVEAT (stated upfront, not hidden): TF32 does NOT change
# tensor dtype — tensors remain torch.float32 even with TF32 enabled.
# PyTorch's flash SDPA backend dispatches strictly on dtype being
# float16/bfloat16, so this probe is expected to be REJECTED. We test
# it anyway for a proven, evidence-based answer rather than assuming.
# ═══════════════════════════════════════════════════════════════════════
_probe_dm = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
    eval_batch_size=1, tokenizer=runner.tokenizer, max_charge=MODEL_MAX_CHARGE, n_workers=0)
_probe_dm.setup(stage='test', annotated=False)
_probe_batch = next(iter(_probe_dm.predict_dataloader()))
_probe_mz, _probe_it, _probe_pr, _ = model._process_batch(_probe_batch)
_pb_mz, _pb_it, _pb_pr = _probe_mz.to(DEVICE), _probe_it.to(DEVICE), _probe_pr.to(DEVICE)
print(f'Probe tensor dtype: {_pb_mz.dtype}  (TF32 does not change this — stays float32)')

_pz = torch.zeros((1, model.max_peptide_len), dtype=torch.long, device=DEVICE)
TF32_FLASH_ACCEPTED = False
_tf32_flash_err = None
try:
    with torch.no_grad():   # NOTE: no autocast — plain FP32+TF32, no BF16
        with sdpa_kernel(backends=[SDPBackend.FLASH_ATTENTION]):
            _pme, _pmk = model.encoder(_pb_mz, _pb_it)
            model.decoder(tokens=_pz, memory=_pme,
                          memory_key_padding_mask=_pmk, precursors=_pb_pr)
    TF32_FLASH_ACCEPTED = True
except RuntimeError as _e:
    _tf32_flash_err = str(_e)[:150]
_sync()

print('\n── FlashAttention + TF32 compatibility probe ─────────────────')
if TF32_FLASH_ACCEPTED:
    print('  RESULT: ACCEPTED — unexpected; investigate PyTorch version behavior.')
else:
    print(f'  RESULT: REJECTED — {_tf32_flash_err}')
    print('  EXPLANATION: TF32 is a GPU matmul execution mode, not a tensor dtype.')
    print('  Tensors remain torch.float32 with TF32 enabled, and FlashAttention\'s')
    print('  SDPA backend requires dtype to literally be float16 or bfloat16.')
    print('  TF32 and FlashAttention are therefore orthogonal — TF32 accelerates')
    print('  GEMMs (aten::addmm/aten::linear) but cannot activate flash attention.')
print('────────────────────────────────────────────────────────────────')

subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

Probe tensor dtype: torch.float32  (TF32 does not change this — stays float32)

── FlashAttention + TF32 compatibility probe ─────────────────
  RESULT: REJECTED — No available kernel. Aborting execution.
  EXPLANATION: TF32 is a GPU matmul execution mode, not a tensor dtype.
  Tensors remain torch.float32 with TF32 enabled, and FlashAttention's
  SDPA backend requires dtype to literally be float16 or bfloat16.
  TF32 and FlashAttention are therefore orthogonal — TF32 accelerates
  GEMMs (aten::addmm/aten::linear) but cannot activate flash attention.
────────────────────────────────────────────────────────────────


In [13]:
# ═══════════════════════════════════════════════════════════════════════
# TF32 CELL 4 — FP32+TF32 + torch.compile
# Fresh compiled wrappers (separate from earlier BF16 compiled_encoder/
# compiled_decoder) so TorchDynamo's guards specialize cleanly on FP32
# dtype rather than sharing cache entries with the BF16 variant.
# Reuses pad_or_select_to_fixed_peaks / _mark_step from earlier session.
# ═══════════════════════════════════════════════════════════════════════
torch._dynamo.reset()  # clear any stale guards from the earlier BF16 compile

compiled_encoder_tf32 = torch.compile(model.encoder, mode='reduce-overhead')
compiled_decoder_tf32 = torch.compile(model.decoder, mode='reduce-overhead')
print('compiled_encoder_tf32 / compiled_decoder_tf32 created (mode=reduce-overhead)')

_gpu_s, _gpu_stop = _start_gpu_monitor()
timing_tf32_compiled = {}

for bs in BATCH_SIZES:
    print(f'\n══ TF32+Compiled  batch_size={bs:4d} ══')
    _dm_bs = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
                              eval_batch_size=bs, tokenizer=runner.tokenizer,
                              max_charge=MODEL_MAX_CHARGE, n_workers=0)
    _dm_bs.setup(stage='test', annotated=False)

    print(f'  Compiling + capturing CUDA Graph for bs={bs}…')
    _t0c = time.perf_counter()
    with torch.no_grad():
        for _w, _wb in enumerate(iter(_dm_bs.predict_dataloader())):
            if _w >= N_WARMUP_BATCHES: break
            _wm, _wi, _wp, _ = model._process_batch(_wb)
            _wm=_wm.to(DEVICE); _wi=_wi.to(DEVICE); _wp=_wp.to(DEVICE)
            _wm, _wi, _ = pad_or_select_to_fixed_peaks(_wm, _wi)
            _mark_step()
            _wme, _wmk = compiled_encoder_tf32(_wm, _wi)
            compiled_decoder_tf32(tokens=torch.zeros((_wm.shape[0], model.max_peptide_len),
                                  dtype=torch.long, device=DEVICE),
                                  memory=_wme, memory_key_padding_mask=_wmk, precursors=_wp)
    _sync()
    compile_s = time.perf_counter() - _t0c
    print(f'  Compile+capture done in {compile_s:.1f}s (excluded from timings below)')

    _t = {k: [] for k in ['fetch','h2d','enc','nar','write','total','tp']}
    _loader_iter = iter(_dm_bs.predict_dataloader()); n_spec = 0
    pbar = tqdm(total=N_TIMING_SPECTRA, desc=f'  bs={bs}', unit='spec')
    while n_spec < N_TIMING_SPECTRA:
        _sync(); t0 = time.perf_counter()
        try: batch = next(_loader_iter)
        except StopIteration: _loader_iter = iter(_dm_bs.predict_dataloader()); batch = next(_loader_iter)
        t_fetch = (time.perf_counter()-t0)*1000

        _sync(); t0 = time.perf_counter()
        mzs, ints, precs, _ = model._process_batch(batch)
        mzs=mzs.to(DEVICE); ints=ints.to(DEVICE); precs=precs.to(DEVICE)
        mzs, ints, _nt = pad_or_select_to_fixed_peaks(mzs, ints)
        _sync(); t_h2d = (time.perf_counter()-t0)*1000; ab = mzs.shape[0]

        with torch.no_grad():
            _mark_step()
            _sync(); t0 = time.perf_counter()
            mem, mmk = compiled_encoder_tf32(mzs, ints)
            _sync(); t_enc = (time.perf_counter()-t0)*1000
            zt = torch.zeros((ab, model.max_peptide_len), dtype=torch.long, device=DEVICE)
            _sync(); t0 = time.perf_counter()
            scores = compiled_decoder_tf32(tokens=zt, memory=mem,
                                           memory_key_padding_mask=mmk, precursors=precs)
            _sync(); t_nar = (time.perf_counter()-t0)*1000

        t0 = time.perf_counter()
        pred = scores.argmax(dim=-1).cpu(); _ = [{'tokens': t.tolist()} for t in pred]
        t_write = (time.perf_counter()-t0)*1000

        tt = t_fetch+t_h2d+t_enc+t_nar+t_write
        for k,v in zip(['fetch','h2d','enc','nar','write','total','tp'],
                       [t_fetch/ab,t_h2d/ab,t_enc/ab,t_nar/ab,t_write/ab,tt/ab,ab/(tt/1000)]):
            _t[k].append(v)
        n_spec += ab; pbar.update(ab)
        if n_spec >= N_TIMING_SPECTRA: break
    pbar.close()

    p = lambda a,q: float(np.percentile(a,q))
    timing_tf32_compiled[bs] = {'n_spec':n_spec,'compile_s':compile_s,
        'fetch':np.mean(_t['fetch']),'h2d':np.mean(_t['h2d']),
        'enc':np.mean(_t['enc']),'nar':np.mean(_t['nar']),'write':np.mean(_t['write']),
        'total':np.mean(_t['total']),'p50':p(_t['total'],50),'p95':p(_t['total'],95),
        'tp':np.mean(_t['tp']),'raw':_t}
    s = timing_tf32_compiled[bs]
    spd = timing_fp32[bs]['total'] / max(s['total'], 0.001)
    print(f'  total={s["total"]:.2f}ms  enc={s["enc"]:.2f}ms  nar={s["nar"]:.2f}ms  '
          f'tp={s["tp"]:.1f}spec/s  vs FP32(no TF32): {spd:.2f}×')
    if DEVICE == 'cuda': torch.cuda.empty_cache()

_gpu_stop.set(); time.sleep(1.0)
gpu_util_tf32c = np.mean([s[0] for s in _gpu_s]) if _gpu_s else 0
gpu_vram_tf32c = np.max([s[1] for s in _gpu_s]) if _gpu_s else 0

u1 = timing_tf32_compiled[1]; ru = u1['raw']
df_stage_tf32c = pd.DataFrame([
    {'Stage':'DataLoader fetch','mean_ms':u1['fetch'],'p50_ms':np.percentile(ru['fetch'],50),'p95_ms':np.percentile(ru['fetch'],95)},
    {'Stage':'H2D+fixed-peak pad','mean_ms':u1['h2d'],'p50_ms':np.percentile(ru['h2d'],50),'p95_ms':np.percentile(ru['h2d'],95)},
    {'Stage':'Encoder(TF32+Compiled)','mean_ms':u1['enc'],'p50_ms':np.percentile(ru['enc'],50),'p95_ms':np.percentile(ru['enc'],95)},
    {'Stage':'Decoder(TF32+Compiled)','mean_ms':u1['nar'],'p50_ms':np.percentile(ru['nar'],50),'p95_ms':np.percentile(ru['nar'],95)},
    {'Stage':'Output write','mean_ms':u1['write'],'p50_ms':np.percentile(ru['write'],50),'p95_ms':np.percentile(ru['write'],95)},
    {'Stage':'TOTAL','mean_ms':u1['total'],'p50_ms':u1['p50'],'p95_ms':u1['p95']},
]).round(3)
df_tp_tf32c = pd.DataFrame([{'batch_size':bs,'total_ms':timing_tf32_compiled[bs]['total'],
    'tp_spec_s':timing_tf32_compiled[bs]['tp'],'enc_ms':timing_tf32_compiled[bs]['enc'],
    'nar_ms':timing_tf32_compiled[bs]['nar'],'compile_s':round(timing_tf32_compiled[bs]['compile_s'],1),
    'vs_fp32':f"{timing_fp32[bs]['total']/max(timing_tf32_compiled[bs]['total'],0.001):.2f}x"}
    for bs in BATCH_SIZES]).round(3)

print(f'\n── Stage breakdown (TF32+Compiled, bs=1) ──\n{df_stage_tf32c.to_string(index=False)}')
print(f'\n── Throughput (TF32+Compiled) ──\n{df_tp_tf32c.to_string(index=False)}')
print(f'\nGPU util: {gpu_util_tf32c:.0f}%  |  Peak VRAM: {gpu_vram_tf32c:.2f} GB')
print(f'10ms target (bs=1): {_tgt(u1["total"])}')
_save(df_stage_tf32c, 'tf32_compiled_stage_bs1.csv')
_save(df_tp_tf32c, 'tf32_compiled_throughput.csv')

compiled_encoder_tf32 / compiled_decoder_tf32 created (mode=reduce-overhead)

══ TF32+Compiled  batch_size=   1 ══


subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Compiling + capturing CUDA Graph for bs=1…
  Compile+capture done in 17.0s (excluded from timings below)


  bs=1: 100%|██████████| 5000/5000 [01:10<00:00, 71.00spec/s]


  total=13.70ms  enc=9.68ms  nar=2.14ms  tp=73.9spec/s  vs FP32(no TF32): 1.54×

══ TF32+Compiled  batch_size=   8 ══


...

subset_profile.mgf: 0 spectra [00:00, ? spectra/s]

  Compiling + capturing CUDA Graph for bs=512…
  Compile+capture done in 8.9s (excluded from timings below)


  bs=512: 5120spec [00:06, 845.61spec/s]                        


  total=1.18ms  enc=0.29ms  nar=0.69ms  tp=849.0spec/s  vs FP32(no TF32): 1.72×

── Stage breakdown (TF32+Compiled, bs=1) ──
                 Stage  mean_ms  p50_ms  p95_ms
      DataLoader fetch    1.379   1.319   1.870
    H2D+fixed-peak pad    0.354   0.358   0.539
Encoder(TF32+Compiled)    9.681   9.206  13.916
Decoder(TF32+Compiled)    2.140   2.108   2.447
          Output write    0.144   0.138   0.191
                 TOTAL   13.698  13.132  18.696

── Throughput (TF32+Compiled) ──
 batch_size  total_ms  tp_spec_s  enc_ms  nar_ms  compile_s vs_fp32
          1    13.698     73.944   9.681   2.140       17.0   1.54x
          8     2.193    460.913   1.231   0.536       18.0   1.43x
         32     1.038    969.144   0.336   0.445       15.8   1.79x
        128     1.106    906.495   0.249   0.629        7.2   1.72x
        512     1.179    849.020   0.292   0.686        8.9   1.72x

GPU util: 21%  |  Peak VRAM: 3.15 GB
10ms target (bs=1): FAILS (13.7ms)
Saved: /teamspace/studio

'/teamspace/studios/this_studio/profiling_after_depthcharge_changes/results/tf32_compiled_throughput.csv'

In [14]:
# ═══════════════════════════════════════════════════════════════════════
# TF32 CELL 5 — torch.profiler comparison (bs=1, 50 profiled spectra)
# Reuses _prof_batches / _prof_batches_fixed from the earlier session's
# Cell 7 (torch.profiler cell) if present; otherwise rebuilds them.
# ═══════════════════════════════════════════════════════════════════════
if '_prof_batches' not in dir():
    print('Pre-fetching profiler batches (not found from earlier session)…')
    _dm_prof = DeNovoDataModule(lance_dir=LANCE_DIR, test_paths=[SUBSET_MGF],
        eval_batch_size=1, tokenizer=runner.tokenizer, max_charge=MODEL_MAX_CHARGE, n_workers=0)
    _dm_prof.setup(stage='test', annotated=False)
    _prof_batches, _prof_batches_fixed = [], []
    N_PROF = PROF_WARMUP + PROF_ACTIVE
    for _b in _dm_prof.predict_dataloader():
        _mz2, _it2, _pr2, _ = model._process_batch(_b)
        _prof_batches.append((_mz2.to(DEVICE), _it2.to(DEVICE), _pr2.to(DEVICE)))
        _mzf, _itf, _ = pad_or_select_to_fixed_peaks(_mz2.to(DEVICE), _it2.to(DEVICE))
        _prof_batches_fixed.append((_mzf, _itf, _pr2.to(DEVICE)))
        if len(_prof_batches) >= N_PROF: break
    while len(_prof_batches) < N_PROF:
        _prof_batches.extend(_prof_batches[:N_PROF-len(_prof_batches)])
        _prof_batches_fixed.extend(_prof_batches_fixed[:N_PROF-len(_prof_batches_fixed)])
    print(f'Using {len(_prof_batches)} batches')

_zt_prof_tf32 = torch.zeros((1, model.max_peptide_len), dtype=torch.long, device=DEVICE)

def _run_prof_tf32(label, trace_path, txt_path, batches, enc_fn, dec_fn,
                    use_mark_step=False, n_warm=10):
    with torch.no_grad():
        for _mz2,_it2,_pr2 in batches[:n_warm]:
            if use_mark_step: _mark_step()
            _me,_mk = enc_fn(_mz2,_it2)
            dec_fn(tokens=_zt_prof_tf32, memory=_me, memory_key_padding_mask=_mk, precursors=_pr2)
    _sync()
    _store = {}
    def _on_ready(p):
        p.export_chrome_trace(trace_path)
        _store['tbl']  = p.key_averages().table(sort_by='cpu_time_total', row_limit=12)
        _store['avgs'] = p.key_averages()
    with profile(activities=ACTS, record_shapes=True,
                 schedule=schedule(wait=0, warmup=PROF_WARMUP, active=PROF_ACTIVE),
                 on_trace_ready=_on_ready) as p:
        with torch.no_grad():
            for _mz2,_it2,_pr2 in batches:
                if use_mark_step: _mark_step()
                with record_function(label):
                    _me,_mk = enc_fn(_mz2,_it2)
                    dec_fn(tokens=_zt_prof_tf32, memory=_me,
                           memory_key_padding_mask=_mk, precursors=_pr2)
                _sync(); p.step()
    print(_store.get('tbl','(no data)'))
    with open(txt_path,'w') as f:
        f.write(f'{label}  bs=1  warmup={PROF_WARMUP}  active={PROF_ACTIVE}\n')
        f.write('='*64+'\n'+str(_store.get('tbl','no data')))
    print(f'Chrome trace → {trace_path}')
    if DEVICE=='cuda': torch.cuda.synchronize(); torch.cuda.empty_cache()
    return _store

# B') FP32+TF32 eager
print('\n── B\') torch.profiler: FP32+TF32 (eager) ────────────────────')
_store_tf32 = _run_prof_tf32('fp32_tf32_forward',
    os.path.join(RESULTS_DIR, 'trace_tf32.json'),
    os.path.join(RESULTS_DIR, 'profiler_tf32.txt'),
    _prof_batches, model.encoder, model.decoder, use_mark_step=False)
_attn_tf32 = _detect_attention_kernel(_store_tf32, "B') FP32+TF32")
_nkern_tf32 = _launch_count(_store_tf32)

# C') TF32 + Compiled
print('\n── C\') torch.profiler: TF32 + Compiled ───────────────────────')
_store_tf32c = _run_prof_tf32('tf32_compiled_forward',
    os.path.join(RESULTS_DIR, 'trace_tf32_compiled.json'),
    os.path.join(RESULTS_DIR, 'profiler_tf32_compiled.txt'),
    _prof_batches_fixed, compiled_encoder_tf32, compiled_decoder_tf32, use_mark_step=True)
_attn_tf32c = _detect_attention_kernel(_store_tf32c, "C') TF32+Compiled")
_nkern_tf32c = _launch_count(_store_tf32c)
_nreg_tf32c  = _compiled_region_count(_store_tf32c)

print('\n── Profiler Summary (TF32 variants) ──────────────────────────')
print(f'{"Metric":<35} {"FP32+TF32":>14} {"TF32+Compiled":>16}')
print('-'*68)
print(f'{"Kernel launches/spectrum":<35} {str(_nkern_tf32):>14} {str(_nkern_tf32c):>16}')
print(f'{"Torch-Compiled Regions":<35} {"N/A":>14} {str(_nreg_tf32c):>16}')
print(f'{"Attention kernel":<35} {"efficient":>14} {"efficient":>16}')
print('(Attention stays memory-efficient in both — TF32 does not change')
print(' tensor dtype, so FlashAttention eligibility is unaffected either way.)')


── B') torch.profiler: FP32+TF32 (eager) ────────────────────
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         0.35%       7.758ms       100.00%        2.214s      44.289ms       0.000us         0.00%     107.683ms       2.154ms            50  
                                      fp32_tf32_forward        22.73%     503.334ms        99.57%        2.205s      44.099ms       0.000us     

In [15]:
# ═══════════════════════════════════════════════════════════════════════
# TF32 CELL 6 — Comparison Plots (adds TF32 variants to existing charts)
# ═══════════════════════════════════════════════════════════════════════
COLORS_TF32 = {'fp32': '#D85A30', 'tf32': '#E8A33D', 'tf32c': '#8E44AD'}
LABELS_TF32 = {'fp32': 'FP32 (baseline)', 'tf32': 'FP32+TF32 (eager)', 'tf32c': 'TF32+Compiled'}
_xi   = list(range(len(BATCH_SIZES)))
_xlbl = [str(b) for b in BATCH_SIZES]

_tps = {'fp32': [timing_fp32[bs]['tp'] for bs in BATCH_SIZES],
        'tf32': [timing_tf32[bs]['tp'] for bs in BATCH_SIZES],
        'tf32c':[timing_tf32_compiled[bs]['tp'] for bs in BATCH_SIZES]}
_lats = {'fp32': [timing_fp32[bs]['total'] for bs in BATCH_SIZES],
         'tf32': [timing_tf32[bs]['total'] for bs in BATCH_SIZES],
         'tf32c':[timing_tf32_compiled[bs]['total'] for bs in BATCH_SIZES]}

fig, (ax_tp, ax_lat) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('TF32 vs FP32 Baseline — Throughput & Latency', fontweight='bold')
for key in ['fp32', 'tf32', 'tf32c']:
    ax_tp.plot(_xi, _tps[key], 'o-', color=COLORS_TF32[key], lw=2, ms=8, label=LABELS_TF32[key])
    ax_lat.plot(_xi, _lats[key], 'o-', color=COLORS_TF32[key], lw=2, ms=8, label=LABELS_TF32[key])
for ax, ylabel, title in [(ax_tp,'Throughput (spec/s)','Throughput vs Batch Size'),
                           (ax_lat,'ms/spectrum','Latency vs Batch Size')]:
    ax.set_xticks(_xi); ax.set_xticklabels(_xlbl)
    ax.set_xlabel('Batch size'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=8, frameon=False); ax.spines[['top','right']].set_visible(False)
ax_lat.axhline(10, color='black', lw=1.5, ls=':', label='10ms target')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'tf32_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {os.path.join(RESULTS_DIR, 'tf32_comparison.png')}")

Saved: /teamspace/studios/this_studio/profiling_after_depthcharge_changes/results/tf32_comparison.png


In [16]:
# ═══════════════════════════════════════════════════════════════════════
# TF32 CELL 7 — Summary Report (answers mentor's TF32 question directly)
# ═══════════════════════════════════════════════════════════════════════
_now = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')

summary_tf32 = f"""CASANOVO NAR — TF32 EXPERIMENT
Generated: {_now}

QUESTION: Is TF32 compatible with FlashAttention and torch.compile, to
avoid BF16's casting overhead?

ANSWER:
  FlashAttention : {'ACCEPTED (unexpected)' if TF32_FLASH_ACCEPTED else 'NOT compatible — confirmed empirically'}
  torch.compile  : Compatible — TF32+Compiled ran successfully, {_nreg_tf32c} compiled region(s)

  TF32 does not change tensor dtype (stays torch.float32) and requires no
  autocast context. FlashAttention's SDPA backend requires dtype to be
  float16/bfloat16 — TF32 tensors don't qualify, so flash attention
  cannot be triggered by TF32 alone. TF32 accelerates the GEMM/matmul
  layer (aten::addmm, aten::linear) directly at the CUDA-core level,
  with zero casting overhead — that overhead was BF16-specific.

TIMING RESULTS (bs=1, real spectra)
  FP32 baseline    : {timing_fp32[1]['total']:.2f} ms  ({timing_fp32[1]['tp']:.1f} spec/s)
  FP32+TF32        : {timing_tf32[1]['total']:.2f} ms  ({timing_tf32[1]['tp']:.1f} spec/s)  [{timing_fp32[1]['total']/max(timing_tf32[1]['total'],0.001):.2f}x vs FP32]
  TF32+Compiled    : {timing_tf32_compiled[1]['total']:.2f} ms  ({timing_tf32_compiled[1]['tp']:.1f} spec/s)  [{timing_fp32[1]['total']/max(timing_tf32_compiled[1]['total'],0.001):.2f}x vs FP32]

FULL BATCH-SIZE COMPARISON
{"Batch":>6} {"FP32(ms)":>10} {"FP32+TF32(ms)":>15} {"TF32+Compiled(ms)":>18}
{'-'*52}
"""
for bs in BATCH_SIZES:
    summary_tf32 += (f'{bs:>6} {timing_fp32[bs]["total"]:>10.2f} '
                      f'{timing_tf32[bs]["total"]:>15.2f} '
                      f'{timing_tf32_compiled[bs]["total"]:>18.2f}\n')

summary_tf32 += f"""
KERNEL LAUNCHES (bs=1, 50 profiled spectra)
  FP32+TF32      : {_nkern_tf32} launches/spectrum
  TF32+Compiled  : {_nkern_tf32c} launches/spectrum

GPU UTILIZATION
  FP32+TF32      : {gpu_util_tf32:.0f}% util | {gpu_vram_tf32:.2f} GB VRAM
  TF32+Compiled  : {gpu_util_tf32c:.0f}% util | {gpu_vram_tf32c:.2f} GB VRAM

ARTIFACTS
  {os.path.join(RESULTS_DIR, 'tf32_comparison.png')}
  {os.path.join(RESULTS_DIR, 'tf32_stage_bs1.csv')} | tf32_throughput.csv
  {os.path.join(RESULTS_DIR, 'tf32_compiled_stage_bs1.csv')} | tf32_compiled_throughput.csv
  trace_tf32.json | trace_tf32_compiled.json
"""
print(summary_tf32)
with open(os.path.join(RESULTS_DIR, 'tf32_summary.txt'), 'w') as f:
    f.write(summary_tf32)
print(f"\nSaved: {os.path.join(RESULTS_DIR, 'tf32_summary.txt')}")

CASANOVO NAR — TF32 EXPERIMENT
Generated: 2026-07-06 17:22

QUESTION: Is TF32 compatible with FlashAttention and torch.compile, to
avoid BF16's casting overhead?

ANSWER:
  FlashAttention : NOT compatible — confirmed empirically
  torch.compile  : Compatible — TF32+Compiled ran successfully, 2 compiled region(s)

  TF32 does not change tensor dtype (stays torch.float32) and requires no
  autocast context. FlashAttention's SDPA backend requires dtype to be
  float16/bfloat16 — TF32 tensors don't qualify, so flash attention
  cannot be triggered by TF32 alone. TF32 accelerates the GEMM/matmul
  layer (aten::addmm, aten::linear) directly at the CUDA-core level,
  with zero casting overhead — that overhead was BF16-specific.

TIMING RESULTS (bs=1, real spectra)
  FP32 baseline    : 21.13 ms  (47.9 spec/s)
  FP32+TF32        : 25.80 ms  (39.2 spec/s)  [0.82x vs FP32]
  TF32+Compiled    : 13.70 ms  (73.9 spec/s)  [1.54x vs FP32]

FULL BATCH-SIZE COMPARISON
 Batch   FP32(ms)   FP32+TF32(ms)  